# Pillar3d matrix run **mG_dec_lam0.014**  (λ=0.014, aux_T=0.5, weighted=on)

Corpus: **DECISIVE (min_margin 0.05)**.  Tests the 'use them all + sharpen' thesis (see the run matrix). The
teacher is widened (top_k=300 + Dirichlet) so low-margin soft-visit targets carry exploration mass;
`--aux-target-temperature 0.5` extracts move-quality. Recipe otherwise = warm-start
pillar3b_epoch_20, lr 5e-5, **aux-warmup 0.5** (shortened from 2.0 so full λ acts from ep1), 6 epochs.

**Upload to `MyDrive/alphatrain/`:** `colorlines_pillar3d_v2.tar.gz`, **`corrections_corpus_mm05.pt`**,
`v13_pillar3a.pt.gz`, `pillar3b_epoch_20.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time
DRIVE = '/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True); os.makedirs('/content/crisis', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3b_epoch_20.pt', '/content/alphatrain/data/pillar3b_epoch_20.pt')
shutil.copy(f'{DRIVE}/corrections_corpus_mm05.pt', '/content/crisis/corrections_corpus.pt')
import torch
_c = torch.load('/content/crisis/corrections_corpus.pt')
print('corpus anchors', _c['boards'].shape[0], _c['_stats'])
assert 9000 < _c['boards'].shape[0] < 60000 and _c['_stats']['min_margin'] >= 0.05, \
    "unexpected corpus — upload corrections_corpus_mm05.pt"
t0 = time.time()
!gzip -dc {DRIVE}/v13_pillar3a.pt.gz > /content/alphatrain/data/v13_pillar3a.pt
print(f"V13 ({time.time()-t0:.0f}s)")
%cd /content
!pip install -q numpy numba scipy

In [ ]:
import torch
print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Train mG_dec_lam0.014 (~12h)

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v13_pillar3a.pt --amp --compile \
    --resume alphatrain/data/pillar3b_epoch_20.pt --warm-start \
    --epochs 6 --batch-size 32768 --lr 5e-5 --warmup-epochs 1 \
    --target-temperature 0.5 \
    --aux-corrections-corpus crisis/corrections_corpus.pt --aux-weighted \
    --aux-lambda 0.014 --aux-target-temperature 0.5 \
    --aux-holdout-frac 0.15 --aux-split-seed 0 \
    --aux-batch-size 256 --aux-warmup-epochs 0.5 --aux-preflight-every 200 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar3d_mG_dec_lam0.014_best.pt \
    --save-dir /content/checkpoints/pillar3d_mG_dec_lam0.014 2>&1 | tee /content/pillar3d_mG_dec_lam0.014_train.log

In [ ]:
import glob, shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar3d_mG_dec_lam0.014/epoch_*.pt')):
    dst = f'{DRIVE}/pillar3d_mG_dec_lam0.014_{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('copied', os.path.basename(f))
shutil.copy('/content/pillar3d_mG_dec_lam0.014_train.log', f'{DRIVE}/pillar3d_mG_dec_lam0.014_train.log')

## Eval — 5,000 games on the CANONICAL list 775000..779999, **`scripts.eval_policy`** (fp16, batch 256)
Uses `scripts.eval_policy` (single-process batched policy player, constant GPU batch, no workers/IPC) —
prints its own progress + full percentile stats. fp16 is fine: scores are reproducible for a fixed
(seed-list, batch), so comparing models on the SAME (775000..779999, batch 256) is clean — the per-game
fp16 noise averages out over 5k games. The bug that bit us was the LIST (775k vs 777k), not the precision.
Needs the top-level `scripts/` package in the tarball (eval_policy.py, batched_rollout.py, __init__.py).
NOTE: the old 16,504 bar was `eval_parallel` — NOT directly comparable. Re-establish the bar by running
the 13.8k mC checkpoint through `eval_policy` on these same seeds+batch, then judge mH/mI against that.
Judge on **MEDIAN (P50) + mean**; floor (<1000) = no-regress guardrail. Sweep ep2/3/4/5.

In [ ]:
%cd /content
for EP in [2, 3, 4, 5]:
    m = f'/content/checkpoints/pillar3d_mG_dec_lam0.014/epoch_{EP}.pt'
    if not os.path.exists(m): continue
    print(f'===== mG_dec_lam0.014 epoch {EP}  (5k 775000..779999) =====', flush=True)
    !python -m scripts.eval_policy --model {m} \
        --seed-start 775000 --seed-end 779999 --device cuda --batch 256